
## 2. Version 1
Sans les préférences, les contraintes de repos et de succession de shifts.

### 2.1. Modélisation mathématique
#### 2.1.1 Ensembles

* L'ensemble des employés $I$ : $\{E01, E02, \ldots, E20\}$
* L'ensemble des shifts $J$ : $\{S01, S02, \ldots, S21\}$
* L'ensemble des compétences $K$ : $\{A, B, C, D\}$

#### 2.1.2 Variables
On modélise le fait que l'employé $i$ soit affecté au shift $j$ par $x_{i,j} \in \{0,1\}$.

#### 2.1.3 Constantes

* La disponibilité de l'employé $i$ pour le shift $j$ :
  $d_{i,j} \in \{0,1\}$

* Le nombre de personnes requises pour la compétence $k$ du shift $j$ :
  $n_{j,k} \in \{0,1,2\}$

* Le nombre de personnes requises sur chaque shift $j$ : 
 $nb\_personnes_{j}$

* L'employé $i$ possède la compétence $k$ (ou non) :
  $c_{i,k} \in \{0,1\}$

* Le nombre de shifts maximum de l'employé $i$ :
  $s_{max,i} \in \{7,8,9,10\}$

* Le nombre de shifts minimum de l'employé $i$ :
  $s_{min,i} \in \{4,5,6\}$


#### 2.1.4 Contraintes
* Pour chaque shift on a le bon nombre d'employés : pour chaque $j \in J$, $\sum_{i \in I} x_{i,j} = nb\_personnes_{j}$

* Pour chaque shift on a assez d'employés par compétence : pour $j \in J$ et $k \in K$, $\sum_{i \in I} x_{i,j} c_{i,k} \geq n_{j,k}$

* Les contraintes sur les shifts max et min : pour $i \in I$, $s_{min,i} \leq \sum_{j \in J} x_{i,j} \leq s_{max,i}$
C'est une contrainte forte pour nous pas souple. 

* Un employé indisponible n'est pas sur le shift  : pour $i \in I$ et $j \in J$, $x_{i,j} \leq d_{i,j}$

#### 2.1.5 Fonction objectif
Il n'y en a pas ici. On étudie déjà la faisabilité.


### 2.2 Résolution


In [1]:
import Pkg
Pkg.activate("..")

using JuMP
using HiGHS

include("../src/donnees.jl")

(; I, J, K, T, S,
   D, N, nb_personnes, C, S_max, S_min,
   TypeShift, Jour, Succ,
   PrefType, PrefJour, PoidsPref,
   TypeShiftBool, JourShiftBool) = charger_donnees();

  Activating project at `C:\Dev\workforce-scheduling-optimization`


In [2]:
model = Model(HiGHS.Optimizer)

# Les variables
@variable(model,x[i in I,j in J], Bin)

# les contraintes
for (p, j) in enumerate(J)
	# Nombre total de personnes sur le shift
	@constraint(
		model,
		sum(x[i,j] for i in I) == nb_personnes[p]
	)		

	# Nombre de personnes possédant chaque compétence
	for k in eachindex(K)
		@constraint(
			model,
			sum( x[I[m],j]*C[m,k] for m in eachindex(I)) >= N[p,k]
		)
	end
end

# Contraintes min / max
for (m,i) in enumerate(I)
		@constraint(model, sum(x[i,j] for j in J) <= S_max[m])
		@constraint(model, sum(x[i,j] for j in J) >= S_min[m])
end

# Contraintes de disponibilité
for (p, j) in enumerate(J)
	for (m,i) in enumerate(I)
		@constraint(
			model,
			x[i,j] <= D[m,p])
	end
end
optimize!(model)
assert_is_solved_and_feasible(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 565 rows; 420 cols; 2604 nonzeros; 420 integer variables (420 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [0e+00, 0e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+01]
Presolving model
136 rows, 248 cols, 1283 nonzeros 0s
111 rows, 246 cols, 1006 nonzeros 0s
Presolve reductions: rows 111(-454); columns 246(-174); nonzeros 1006(-1598) 
Objective function is integral with scale 1

Solving MIP model with:
   111 rows
   246 cols (246 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   1006 nonzeros
   Thread count 8 (of 16 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized

LoadError: The model was not solved correctly. Here is the output of `solution_summary` to help debug why this happened:

solution_summary(; result = 1, verbose = false)
├ solver_name          : HiGHS
├ Termination
│ ├ termination_status : INFEASIBLE
│ ├ result_count       : 1
│ ├ raw_status         : kHighsModelStatusInfeasible
│ └ objective_bound    : 0.00000e+00
├ Solution (result = 1)
│ ├ primal_status        : NO_SOLUTION
│ ├ dual_status          : INFEASIBILITY_CERTIFICATE
│ ├ objective_value      : 0.00000e+00
│ ├ dual_objective_value : 1.00000e+00
│ └ relative_gap         : Inf
└ Work counters
  ├ solve_time (sec)   : 2.05801e-02
  ├ simplex_iterations : 241
  ├ barrier_iterations : 0
  └ node_count         : 0


Avec les contraintes sur les shifts minimum, le problème est infaisable. En effet, il faut 94 shifts si on additionne les shifts minimum mais nous n'avons que 93 shifts disponibles. En enlevant les contraintes sur les shifts minimums, le problème admet des solutions.

## 3. Version 2 
On ajoute les contraintes de repos avec les shifts qui ne peuvent pas se suivre et le shift minimum est une contrainte souple

### 3.1. Modélisation mathématique

#### 3.1.1 Ensembles

* L'ensemble des employés $I$ : $\{E01, E02, \ldots, E20\}$
* L'ensemble des shifts $J$ : $\{S01, S02, \ldots, S21\}$
* L'ensemble des compétences $K$ : $\{A, B, C, D\}$
* L'ensemble des types de shifts $T$ : $\{M, AP, N\}$

#### 3.1.2 Variables

On modélise le fait que l'employé $i$ soit affecté au shift $j$ par $x_{i,j} \in \{0,1\}$.

On ajoute des variables d'écart pour les contraintes souples sur les shifts minimum : $e_i \in \mathbb{N}$ avec $i \in I$.

#### 3.1.3 Constantes

* La disponibilité de l'employé $i$ pour le shift $j$ :
  $d_{i,j} \in \{0,1\}$

* Le nombre de personnes requises pour la compétence $k$ du shift $j$ :
  $n_{j,k} \in \{0,1,2\}$

* Le nombre de personnes requises sur chaque shift $j$ : 
 $nb\_personnes_{j}$

* L'employé $i$ possède la compétence $k$ (ou non) :
  $c_{i,k} \in \{0,1\}$

* Le nombre de shifts maximum de l'employé $i$ :
  $s_{max,i} \in \{7,8,9,10\}$

* Le nombre de shifts minimum de l'employé $i$ :
  $s_{min,i} \in \{4,5,6\}$

* Le type $type_j \in T$ du shift $j$.

* Les successions $succ_{t_1,t_2} \in \{0,1\}$ 
  (0 = succession interdite, 1 = succession autorisée), 
  où $t_1$ est un type de shift et $t_2$ est le type du shift suivant à analyser (qui a lieu le même jour ou le lendemain).

#### 3.1.4 Contraintes

* Pour chaque shift, on a le bon nombre d'employés :
  
  Pour chaque $j \in J$,

  $\sum_{i \in I} x_{i,j} = nb\_personnes_{j}$

* Pour chaque shift, on a assez d'employés par compétence :
  
  Pour $j \in J$ et $k \in K$,

  $\sum_{i \in I} x_{i,j}c_{i,k} \geq n_{j,k}$

* Les contraintes sur les shifts maximum (contrainte dure) :
  
  Pour $i \in I$,

  $\sum_{j \in J} x_{i,j} \leq s_{max,i}$

* Les contraintes sur les shifts minimum (contrainte souple) qu'on pénalisera avec p = 1 :
  
  Pour $i \in I$,

  $s_{min,i} \leq e_i + \sum_{j \in J} x_{i,j}$

* Un employé indisponible n'est pas affecté au shift :
  
  Pour $i \in I$ et $j \in J$,

  $x_{i,j} \leq d_{i,j}$

* Pour un employé $i$, deux shifts successifs ne peuvent être effectués lorsque leur succession est interdite :
  
  Pour $i \in I$ et deux shifts successifs,

  $x_{i,j} + x_{i,j+1}
  \leq 1 + succ_{type_j,type_{j+1}}$


  $x_{i,j} + x_{i,j+2}
  \leq 1 + succ_{type_j,type_{j+2}}$

#### 3.1.5 Fonction objectif
On veut minimiser les variables $e_i$ qui sont pénalisées par $p=1$. 

$\min \sum_{i \in I} p*e_i$


### 3.2 Résolution

In [3]:
model = Model(HiGHS.Optimizer)

# Les variables
@variable(model,x[i in I,j in J], Bin)
@variable(model,e[i in I] >= 0, Int)

# Les contraintes 
for (p, j) in enumerate(J)
	# Nombre total de personnes sur le shift
	@constraint(
		model,
		sum(x[i,j] for i in I) == nb_personnes[p]
	)	

	# Nombre de personnes possédant chaque compétence
	for k in eachindex(K)
		@constraint(
			model,
			sum( x[I[m],j]*C[m,k] for m in eachindex(I)) >= N[p,k]
		)
	end
end

# Contraintes min / max
for (m,i) in enumerate(I)
	@constraint(model, sum(x[i,j] for j in J) <= S_max[m])
	@constraint(model, sum(x[i,j] for j in J) + e[i] >= S_min[m])
end

# Contraintes de disponibilité
for (p, j) in enumerate(J)
	for (m,i) in enumerate(I)
		@constraint(
			model,
			x[i,j] <= D[m,p])
	end
end	

# Contraintes de repos
for i in I
	for (p,j) in enumerate(J)
		if p < length(J)
			@constraint(model, x[i,J[p]] + x[i,J[p+1]] <= 1 + Succ[TypeShift[J[p]], TypeShift[J[p+1]]])
		end
        if p < length(J) - 1
			@constraint(model, x[i,J[p]] + x[i,J[p+2]] <= 1 + Succ[TypeShift[J[p]], TypeShift[J[p+2]]])
		end
	end
end

# Fonction objectif 
@objective(model, Min, sum(e[i] for i in I))

e[E01] + e[E02] + e[E03] + e[E04] + e[E05] + e[E06] + e[E07] + e[E08] + e[E09] + e[E10] + e[E11] + e[E12] + e[E13] + e[E14] + e[E15] + e[E16] + e[E17] + e[E18] + e[E19] + e[E20]

In [4]:
optimize!(model)
assert_is_solved_and_feasible(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 1345 rows; 440 cols; 4184 nonzeros; 440 integer variables (420 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+01]
Presolving model
267 rows, 268 cols, 1565 nonzeros 0s
254 rows, 264 cols, 1496 nonzeros 0s
Presolve reductions: rows 254(-1091); columns 264(-176); nonzeros 1496(-2688) 
Objective function is integral with scale 1

Solving MIP model with:
   254 rows
   264 cols (244 binary, 20 integer, 0 implied int., 0 continuous, 0 domain fixed)
   1496 nonzeros
   Thread count 8 (of 16 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomi

In [5]:
valeur_x = value(x)
valeur_e = value(e)

for i in I
    shifts = [j for j in J if value(x[i,j]) > 0.5]
    println("Employé $i : ", join(shifts, ", "))
end

println("e = ", valeur_e)

Employé E01 : S04, S05, S11, S16, S17
Employé E02 : S01, S07, S13, S16, S17
Employé E03 : S04, S09, S12, S21
Employé E04 : S02, S05, S13, S17
Employé E05 : S02, S10, S13, S14, S19, S20
Employé E06 : S01, S04, S05, S10, S11
Employé E07 : S02, S08, S12, S17
Employé E08 : S03, S06, S15, S21
Employé E09 : S01, S04, S07, S08, S13, S14
Employé E10 : S04, S07, S10, S11, S16
Employé E11 : S01, S07, S13, S19
Employé E12 : S07, S08, S14, S19
Employé E13 : S02, S03, S08, S14, S15, S20
Employé E14 : S03, S12, S15, S18
Employé E15 : S05, S08, S14, S17, S20
Employé E16 : S05, S10, S11, S16
Employé E17 : S08, S09, S14
Employé E18 : S02, S06, S10, S11, S18, S21
Employé E19 : S06, S09, S15, S18
Employé E20 : S01, S04, S13, S19, S20
e = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, ["E01", "E02", "E03", "E04", "E05", "E06", "E07", "E08", "E09", "E10", "E11", "E12", "E13", "E14", "E15", "E16", "E17", "E18", "E19", "E20"]
And data, a 20-element Vector{Float64}:
 0.0
 0.0
 0

In [6]:
[(i, sum(value(x[i,j]) for j in J), value(e[i])) for i in I]

20-element Vector{Tuple{String, Float64, Float64}}:
 ("E01", 5.0, 0.0)
 ("E02", 5.0, 0.0)
 ("E03", 4.0, 0.0)
 ("E04", 4.0, 0.0)
 ("E05", 6.0, 0.0)
 ("E06", 5.0, 0.0)
 ("E07", 4.0, 0.0)
 ("E08", 4.0, 0.0)
 ("E09", 6.0, 0.0)
 ("E10", 5.0, 0.0)
 ("E11", 4.0, 0.0)
 ("E12", 4.0, 0.0)
 ("E13", 6.0, 0.0)
 ("E14", 4.0, 0.0)
 ("E15", 5.0, 0.0)
 ("E16", 4.0, 0.0)
 ("E17", 3.0, 1.0)
 ("E18", 6.0, 0.0)
 ("E19", 4.0, 0.0)
 ("E20", 5.0, 0.0)

Le solveur trouve une solution optimale avec un employé qui a un shift de moins que le nombre espéré.


## 4. Version 3 
Version complète

On va ajouter ici ces contraintes : 
* les préférences des employés
* éviter les semaines trop chargées

Ces contraintes peuvent être violées moyennant une pénalité. En plus de la modélisation, on doit définir quel niveau de pénalité. Peut-être ne doivent-elles pas être toutes équivalentes. 

Je choisis de pénaliser chaque shift qui ne sera pas la préférence de l'employé. Une semaine trop chargée est une semaine de 6 shifts ou plus. Donc elles seront pénalisées. 

### 4.1. Modélisation mathématique

#### 4.1.1 Ensembles

* L'ensemble des employés $I$ : $\{E01, E02, \ldots, E20\}$
* L'ensemble des shifts $J$ : $\{S01, S02, \ldots, S21\}$
* L'ensemble des compétences $K$ : $\{A, B, C, D\}$
* L'ensemble des types de shifts $T$ : $\{M, AP, N\}$
* L'ensemble des jours de la semaine $S$ : $\{lundi, mardi, mercredi, jeudi, vendredi, samedi, dimanche\}$

#### 4.1.2 Variables

On modélise :
* le fait que l'employé $i$ soit affecté au shift $j$ par $x_{i,j} \in \{0,1\}$
* des variables d'écart pour les contraintes souples sur les shifts minimum : $e_i^{min} \in \mathbb{N}$ avec $i \in I$
* des variables de violation des préférences de jour des employés : $v_{i,j}^{jour} \in  \{0,1\}$ avec $i \in I$ et $j \in J$
* des variables de violation des préférences de type de shift des employés : $v_{i,j}^{type} \in  \{0,1\}$ avec $i \in I$ et $j \in J$
* des variables de violation des semaines trop chargées : $v_i^{semaine} \in \mathbb{N}$ avec $i \in I$.

#### 4.1.3 Constantes

* La disponibilité de l'employé $i$ pour le shift $j$ :
  $d_{i,j} \in \{0,1\}$

* Le nombre de personnes requises pour la compétence $k$ du shift $j$ :
  $n_{j,k} \in \{0,1,2\}$

* Le nombre de personnes requises sur chaque shift $j$ : $nb\_personnes_{j}$

* L'employé $i$ possède la compétence $k$ (ou non) :
  $c_{i,k} \in \{0,1\}$

* Le nombre de shifts maximum de l'employé $i$ :
  $s_{max,i} \in \{7,8,9,10\}$

* Le nombre de shifts minimum de l'employé $i$ :
  $s_{min,i} \in \{4,5,6\}$

* Le shift $j$ est de type $type_{j} \in T$.

* Le shift $j$ a lieu le jour $jour_{j} \in S$.


* Les successions $succ_{t_1,t_2} \in \{0,1\}$ 
  (0 = succession interdite, 1 = succession autorisée), 
  où $t_1$ est un type de shift et $t_2$ est le type du shift suivant à analyser (qui a lieu le même jour ou le lendemain).


* L'employé $i$ préfère le type $t$ : $preftype_{i,t} \in \{0,1\}$.

* L'employé $i$ préfère le jour $jr$ : $prefjour_{i,jr} \in \{0,1\}$. 

*  Le poids des préférences des employés $poidspref_i \in \mathbb{N}$.
* On définit $a_{j,t} = 1$ si le shift $j$ est de type $t$, 0 sinon.
* On définit $b_{j,jr} = 1$ si le shift $j$ a lieu le jour $jr$, 0 sinon.

#### 4.1.4 Contraintes

* Pour chaque shift, on a le bon nombre d'employés :
  
  Pour chaque $j \in J$, $\sum_{i \in I} x_{i,j} = nb\_personnes_{j}$

* Pour chaque shift, on a assez d'employés par compétence :
  
  Pour $j \in J$ et $k \in K$, $\sum_{i \in I} x_{i,j}c_{i,k} \geq n_{j,k}$

* Les contraintes sur les shifts maximum (contrainte dure) :
  
  Pour $i \in I$,  $\sum_{j \in J} x_{i,j} \leq s_{max,i}$

* Les contraintes sur les shifts minimum (contrainte souple) qu'on pénalisera avec $p^{min} = 1$ :
  
  Pour $i \in I$, $s_{min,i} \leq e_i^{min} + \sum_{j \in J} x_{i,j}$

* Un employé indisponible n'est pas affecté au shift :
  
  Pour $i \in I$ et $j \in J$, $x_{i,j} \leq d_{i,j}$

* Pour un employé $i$, deux shifts ne peuvent être effectués lorsque leur succession est interdite (comme on modélise sur une unique semaine, on exclut la contrainte du dimanche soir au lundi matin) :
  
  Pour $i \in I$ et $j,j+1 \in J$, $x_{i,j} + x_{i,j+1}
  \leq 1 + succ_{type_j,type_{j+1}}$

  Pour $i \in I$ et $j,j+2 \in J$, $x_{i,j} + x_{i,j+2}
  \leq 1 + succ_{type_j,type_{j+2}}$

* Pour un employé $i$, s'il préfère un shift, pour chaque shift $j$ et chaque type de shift $t$ : $x_{i,j} * a_{j,t} (1-preftype_{i,t}) \leq v_{i,j}^{type}$

* Pour un employé $i$, s'il préfère un jour, pour chaque shift $j$ et chaque jour $jr$ : $x_{i,j} * b_{j,jr} (1-prefjour_{i,jr}) \leq v_{i,j}^{jour}$

* Pour éviter les semaines trop chargées, pour chaque employé $i$, $\sum_{j \in J} x_{i,j} - v_{i}^{semaine} \leq 5$

#### 4.1.5 Fonction objectif
$p^{min}$ et  $p^{semaine}$ sont des paramètres de pénalisation à définir et initialisés à 1. 

$\min   p^{min} \sum_{i \in I} e_i^{min} + p^{semaine}  \sum_{i \in I} v_{i}^{semaine} +   \sum_{i \in I} poidspref_{i} \sum_{j \in J} v_{i,j}^{type} +  \sum_{i \in I} poidspref_{i}  \sum_{j \in J} v_{i,j}^{jour}$

### 4.2 Résolution

In [7]:
model = Model(HiGHS.Optimizer)

# Les variables
@variable(model,x[i in I,j in J], Bin)
@variable(model,e[i in I] >= 0, Int)
@variable(model,viol_jour[i in I, j in J], Bin)
@variable(model,viol_type[i in I, j in J], Bin)
@variable(model,viol_semaine[i in I] >= 0, Int);

In [8]:
# Les contraintes 
for (p, j) in enumerate(J)
	# Nombre total de personnes sur le shift
	@constraint(
		model,
		sum(x[i,j] for i in I) == nb_personnes[p]
	)	

	# Nombre de personnes possédant chaque compétence
	for k in eachindex(K)
		@constraint(
			model,
			sum( x[I[m],j]*C[m,k] for m in eachindex(I)) >= N[p,k]
		)
	end
end

# Contraintes min / max
for (m,i) in enumerate(I)
	@constraint(model, sum(x[i,j] for j in J) <= S_max[m])
	@constraint(model, sum(x[i,j] for j in J) + e[i] >= S_min[m])
end

# Contraintes de disponibilité
for (p, j) in enumerate(J)
	for (m,i) in enumerate(I)
		@constraint(
			model,
			x[i,j] <= D[m,p])
	end
end	

# Contraintes de repos
for i in I
	for (p,j) in enumerate(J)
		if p < length(J)
			@constraint(model, x[i,J[p]] + x[i,J[p+1]] <= 1 + Succ[TypeShift[J[p]], TypeShift[J[p+1]]])
		end
        if p < length(J) - 1
			@constraint(model, x[i,J[p]] + x[i,J[p+2]] <= 1 + Succ[TypeShift[J[p]], TypeShift[J[p+2]]])
		end
	end
end

# Contraintes sur les préférences de shift 
for (p,i) in enumerate(I)
	if  sum(PrefType[p, :]) == 1 #il y a une pref de type
		for (m,j) in enumerate(J) 
			for (l,t) in enumerate(T)
				@constraint(
					model, 
					x[i,j] * TypeShiftBool[m,l] * (1 - PrefType[p,l])
                <= viol_type[i,j])
			end
		end
	end 
end

# Contraintes sur les préférences de jour 
for (p,i) in enumerate(I)
	if  sum(PrefJour[p, :]) == 1 #il y a une pref de jour
		for (m,j) in enumerate(J) 
			for (l,k) in enumerate(S)
				@constraint(
					model, 
					x[i,j] * JourShiftBool[m,l] * (1 - PrefJour[p,l])
                <= viol_jour[i,j])
			end
		end
	end 
end

# Contraintes sur les semaines trop chargées
for i in I
	@constraint(model, sum(x[i,j] for j in J) - viol_semaine[i]  <= 5)
end

p_semaine = 1
p_min = 1

1

In [9]:
# Fonction objectif 
	@objective(model, Min, 
			   p_min * sum(e[i] for i in I)
			   + sum(PoidsPref[i] * sum(viol_type[i,j] for j in J) for i in I)
			  	+ sum(PoidsPref[i] * sum(viol_jour[i,j] for j in J) for i in I)
			   + p_semaine * sum(viol_semaine[i] for i in I)
			   
			  );

In [10]:
optimize!(model)
assert_is_solved_and_feasible(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 2814 rows; 1300 cols; 6351 nonzeros; 1300 integer variables (1260 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 3e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+01]
Presolving model
406 rows, 288 cols, 1833 nonzeros 0s
278 rows, 284 cols, 1795 nonzeros 0s
274 rows, 284 cols, 1764 nonzeros 0s
Presolve reductions: rows 274(-2540); columns 284(-1016); nonzeros 1764(-4587) 
Objective function is integral with scale 1

Solving MIP model with:
   274 rows
   284 cols (244 binary, 40 integer, 0 implied int., 0 continuous, 0 domain fixed)
   1764 nonzeros
   Thread count 8 (of 16 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L

In [11]:
valeur_x = value(x)
valeur_e = value(e)
valeur_type = value(viol_type)
valeur_jour = value(viol_jour)
valeur_semaine = value(viol_semaine);

In [12]:
for i in I
    shifts = [j for j in J if value(x[i,j]) > 0.5]
    println("Employé $i : ", join(shifts, ", "))
end

Employé E01 : S04, S07, S10, S13, S19
Employé E02 : S04, S07, S10, S16, S19
Employé E03 : S08, S12, S17, S20
Employé E04 : S05, S08, S11, S14, S17
Employé E05 : S02, S10, S16, S17, S20
Employé E06 : S01, S07, S13, S16, S19
Employé E07 : S02, S05, S08, S11, S14
Employé E08 : S06, S09, S15, S18
Employé E09 : S02, S08, S14, S20
Employé E10 : S01, S04, S09, S13, S21
Employé E11 : S05, S11, S17
Employé E12 : S01, S04, S10, S13
Employé E13 : S01, S04, S07, S13, S16, S19
Employé E14 : S03, S12, S15, S21
Employé E15 : S02, S05, S08, S11, S14
Employé E16 : S01, S04, S07, S10, S13
Employé E17 : S09, S12, S20, S21
Employé E18 : S03, S06, S11, S14, S15, S18
Employé E19 : S03, S06, S15, S18
Employé E20 : S02, S05, S08, S14, S17


In [13]:
println("Contraintes violées : ")
for i in I
	if valeur_e[i] > 0 
		println("Employé qui fait moins de shifts que souhaité $i : ",  valeur_e[i])
	end
	for j in J
		if valeur_type[i,j] > 0
			println("Type de shift Employé/Shift $i $j : ", valeur_type[i,j])
		end
		
		if valeur_jour[i,j] > 0
			println("Jour Employé/Shift $i $j : ", valeur_jour[i,j])
		end
	end

	if valeur_semaine[i] > 0
		println("Semaine trop chargée Employé $i : ", valeur_semaine[i])
	end
end

Contraintes violées : 
Type de shift Employé/Shift E03 S12 : 1.0
Employé qui fait moins de shifts que souhaité E05 : 1.0
Jour Employé/Shift E05 S02 : 1.0
Jour Employé/Shift E05 S10 : 1.0
Jour Employé/Shift E05 S20 : 1.0
Employé qui fait moins de shifts que souhaité E09 : 2.0
Jour Employé/Shift E10 S01 : 1.0
Jour Employé/Shift E10 S04 : 1.0
Jour Employé/Shift E10 S09 : 1.0
Jour Employé/Shift E10 S21 : 1.0
Employé qui fait moins de shifts que souhaité E11 : 1.0
Semaine trop chargée Employé E13 : 1.0
Jour Employé/Shift E17 S09 : 1.0
Jour Employé/Shift E17 S12 : 1.0
Semaine trop chargée Employé E18 : 1.0


Le modèle admet une solution optimale avec un coût de 18. 

Contraintes violées : 

* Type de shift Employé/Shift E03 S12 : 1.0
* Employé qui fait moins de shifts que souhaité E05 : 1.0
* Jour Employé/Shift E05 S02 : 1.0
* Jour Employé/Shift E05 S10 : 1.0
* Jour Employé/Shift E05 S20 : 1.0
* Employé qui fait moins de shifts que souhaité E09 : 2.0
* Jour Employé/Shift E10 S01 : 1.0
* Jour Employé/Shift E10 S04 : 1.0
* Jour Employé/Shift E10 S09 : 1.0
* Jour Employé/Shift E10 S21 : 1.0
* Employé qui fait moins de shifts que souhaité E11 : 1.0
* Semaine trop chargée Employé E13 : 1.0
* Jour Employé/Shift E17 S09 : 1.0
* Jour Employé/Shift E17 S12 : 1.0
* Semaine trop chargée Employé E18 : 1.0